**Imports**

In [1]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

**Spark Session**

In [ ]:
spark = SparkSession.builder \
    .appName("NYC_Taxi_Preprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

#Kritik hatalar
spark.sparkContext.setLogLevel("ERROR")

print("Spark Oturumu Başarıyla Başlatıldı!")
print(f"Spark Versiyonu: {spark.version}")

**Klasör Yolları**

In [14]:
RAW_DIR = "data/raw"
CLEAN_DIR = "data/clean/yellow_tripdata_2023"

Ay bazında okumak için döngü

In [ ]:
for month in range(1, 13):
    month_str = f"{month:02d}"
    file_name = f"yellow_tripdata_2023-{month_str}.parquet"
    file_path = os.path.join(RAW_DIR, file_name)

    # O aya ait klasör yoksa hata vermeden atla
    if not os.path.exists(file_path):
        print(f"Atlanıyor: {file_path} bulunamadı.")
        continue

    print(f"[{month_str}/12] {file_name} işleniyor...")
    start_time = time.perf_counter()

    # Tek bir ay için parquet dosyasını oku
    df_month = spark.read.parquet(file_path)

    # Projection Prunning
    selected_columns = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'trip_distance', 'total_amount']
    df_filtered = df_month.select(*selected_columns)

    # Yeni sütunlar ekle
    df_filtered = df_filtered.withColumn("pickup_year", F.year(F.col("tpep_pickup_datetime")))\
                             .withColumn("pickup_month", F.month(F.col("tpep_pickup_datetime")))
    
    # Veriyi temizle
    df_cleaned = df_filtered.filter(
        (F.col("pickup_year") == 2023) &
        (F.col("pickup_month") == month) &
        (F.col("tpep_pickup_datetime") < F.col("tpep_dropoff_datetime")) &
        (F.col("total_amount") > 0.0) & (F.col("total_amount") < 500.0) &
        (F.col("trip_distance") > 0.0) & (F.col("trip_distance") < 100.0) &
        (F.col("PULocationID").between(1, 263)) & 
        (F.col("DOLocationID").between(1, 263))
    )

    # Seyahat süresini hesapla ve yeni bir sütun olarak ekle
    df_cleaned = df_cleaned.withColumn(
        "trip_duration_seconds",
        F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")
    ).filter((F.col("trip_duration_seconds") > 30) & (F.col("trip_duration_seconds") < 18000))

    # Duplicate verileri yok et
    df_cleaned = df_cleaned.dropDuplicates()

    # Null değerleri yok et
    df_cleaned = df_cleaned.dropna(subset=["tpep_pickup_datetime", "tpep_dropoff_datetime", 
                                           "PULocationID", "DOLocationID", "trip_distance", 
                                           "total_amount"])
    
    # Temizlenmiş veriyi diske append et ve partiton et
    df_cleaned.write \
        .mode("append") \
        .partitionBy("pickup_year", "pickup_month") \
        .parquet(CLEAN_DIR)
    
    end_time = time.perf_counter()
    print(f"-> {file_name} tamamlandı. Süre: {end_time - start_time:.2f} saniye.\n")

print("YELLOW TAXI VERİ SETİ BAŞARIYLA TEMİZLENDİ VE OPTİMİZE EDİLDİ")
